# 04 · Baselines y evaluación

**Qué hace este notebook:** ajusta la escalera de modelos de referencia, los
compara con un protocolo único y elige uno. Baselines y evaluación van juntos a
propósito: un baseline sin métrica no es nada y una métrica sin varios modelos que
comparar no dice si es buena. Son un solo bucle y separarlos obligaría a que
ambos notebooks conocieran el mismo protocolo de validación, que es la forma
habitual de acabar comparando modelos medidos de formas distintas.

**Lo que sí se ha separado es la interpretabilidad** (`05`). No por longitud, sino
porque responde a otra pregunta -- *por qué* predice esto, no *cuánto* acierta --
se aplica a un solo modelo en lugar de a todos, y es órdenes de magnitud más
lenta. Metida aquí, cada vez que se quisiera añadir un modelo habría que esperar
a que se recalculara SHAP.

| | |
|---|---|
| Lee | `data/gold/model_matrix.parquet`, `reports/selection/selected_features.json` |
| Escribe | `reports/baselines/cv_scores.csv`, `reports/baselines/test_scores.csv` |
| Escribe | `models/<nombre>.joblib` para el modelo elegido |

**La regla que hay que respetar hasta el final:** `test` se toca **una vez**, en
la sección 5, cuando ya no queda ninguna decisión por tomar. Todo lo demás --
comparar modelos, ajustar hiperparámetros, elegir variables -- se hace con
validación cruzada sobre `train`. Un conjunto de test consultado dos veces es un
conjunto de validación con mejor nombre.

In [ ]:
DATASET = "model_matrix.parquet"

# `True` usa la selección de 03; `False` usa todos los predictores, que es la
# comparación honesta contra la que juzgar si seleccionar sirvió de algo.
USE_SELECTED = True

# Particiones de la validación cruzada temporal, y hueco entre ajuste y
# evaluación dentro de cada una. El hueco debe superar la longitud de
# decorrelación medida en §2.3 de 01.
CV_SPLITS = 5
CV_GAP = 24

# Métrica que decide. RMSE castiga los errores grandes, que en oleaje son los
# temporales: exactamente el régimen que importa. MAE sería la elección si lo que
# se buscase fuese el comportamiento típico.
DECIDE_BY = "RMSE"

In [ ]:
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.typing import ArrayLike
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

from packagename import get_settings, set_seed, setup_logging
from packagename.etl import read_table, write_table
from packagename.viz import COLOR_NAMES, NEUTRALS, apply_style, savefig

setup_logging(level="INFO")
apply_style("paper")

settings = get_settings()
seed = set_seed(settings.random_seed)
settings.paths.ensure()

FIG = "baselines"
TABLES = settings.paths.reports / "baselines"
TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
selection = json.loads(
    (settings.paths.reports / "selection" / "selected_features.json").read_text()
)
artefact = joblib.load(settings.paths.models / "preprocessor.joblib")
TARGET = selection["target"]

matrix = read_table(settings.paths.gold / DATASET).set_index("time").sort_index()
predictors = selection["selected"] if USE_SELECTED else artefact["transformed_names"]

train = matrix[matrix["split"] == "train"]
valid = matrix[matrix["split"] == "valid"]
test = matrix[matrix["split"] == "test"]

print(f"{len(predictors)} predictores")
for name, block in (("train", train), ("valid", valid), ("test", test)):
    print(f"{name:6s} {len(block):>8,} filas   {block.index.min()} -> {block.index.max()}")

## 1. Métricas

`RMSE`, `MAE`, `R²` y `Bias` describen cualquier regresión. Las tres siguientes
son las que se usan en validación de modelos de oleaje y conviene entender qué
añaden:

- **SI** (*scatter index*): el RMSE una vez quitado el sesgo, dividido por la
  media observada. Es el error aleatorio en términos relativos, y es la métrica
  que permite comparar un modelo del Cantábrico con uno del Mediterráneo pese a
  que las alturas de ola típicas no sean parecidas.
- **HH** (Hanna y Heinold): normaliza por el producto de observado y predicho en
  lugar de por el observado solo. A diferencia del SI, no premia a un modelo por
  subestimar sistemáticamente, que es el modo de fallo clásico en oleaje.
- **Pearson**: mide la sincronía temporal, es decir, si los temporales se
  predicen *cuando* ocurren. Un modelo puede tener un RMSE decente y una
  correlación mediocre si acierta el nivel medio y llega tarde a los picos.

Las cuatro primeras métricas están en `sklearn`; se implementan aquí de todas
formas para que las siete salgan de una sola pasada sobre los residuos y no haya
dos definiciones del mismo número en circulación.

In [ ]:
# -> src/packagename/models/metrics.py en cuanto 05 necesite las mismas
# definiciones: dos copias de una métrica son dos métricas.
def wave_metrics(observations: ArrayLike, predictions: ArrayLike) -> dict[str, float]:
    observed = np.asarray(observations, dtype=float)
    predicted = np.asarray(predictions, dtype=float)
    residual = predicted - observed
    bias = residual.mean()
    return {
        "RMSE": float(np.sqrt(np.mean(residual**2))),
        "MAE": float(np.mean(np.abs(residual))),
        "R2": float(1 - np.sum(residual**2) / np.sum((observed - observed.mean()) ** 2)),
        "Bias": float(bias),
        # El SI usa el residuo *centrado*: es dispersión, no sesgo.
        "SI": float(np.sqrt(np.mean((residual - bias) ** 2)) / observed.mean()),
        "HH": float(np.sqrt(np.sum(residual**2) / np.sum(observed * predicted))),
        "Pearson": float(np.corrcoef(observed, predicted)[0, 1]),
    }

## 2. La escalera de baselines

En este orden, y el orden es el argumento:

| Modelo | Qué demuestra si no se le gana |
|---|---|
| Media | Que el objetivo no tiene estructura aprovechable. El suelo absoluto. |
| Climatología | Que basta el calendario: el modelo no aporta nada sobre la estacionalidad. |
| Persistencia | Que basta el valor anterior. **El baseline que de verdad hay que batir.** |
| Regresión lineal | Que la relación es lineal y sin regularizar. |
| Ridge / LASSO / Elastic Net | Que la regularización no hacía falta, o cuál de las tres. |
| Random Forest | Que no hay no linealidad que explotar. |
| XGBoost / HistGB | Referencia en datos tabulares. Si el modelo final no le gana, no hay modelo final. |

**Climatología y persistencia no estaban en el plan original y son las dos más
importantes.** Una regresión lineal sobre variables atmosféricas parece un suelo
razonable, pero en una serie con la memoria que mostró la ACF en `01`, predecir
"lo mismo que hace una hora" es difícil de batir. Un R² de 0.9 contra la media es
irrelevante si la persistencia da 0.92: significa que el modelo ha aprendido a
copiar el pasado, y todo el trabajo de las variables atmosféricas no ha servido.
Publicar sin este baseline es la forma más común de sobrevalorar un resultado.

`HistGradientBoostingRegressor` cubre el hueco de LightGBM/CatBoost: viene con
`sklearn`, no necesita `libomp` y sigue la misma idea. Si hace falta el original,
`uv add --group analysis lightgbm catboost`.

In [ ]:
# Los baselines temporales no usan predictores, así que no encajan en la
# interfaz de sklearn y se calculan aparte. Van sobre el objetivo directamente.
def persistence(observed: pd.Series, lag: int = 1) -> pd.Series:
    return observed.shift(lag)


def climatology(reference: pd.Series, index: pd.DatetimeIndex) -> pd.Series:
    # La media por día del año se calcula sólo con train y se proyecta sobre el
    # índice pedido: un baseline que mira su propio target sería tramposo.
    by_doy = reference.groupby(reference.index.dayofyear).mean()
    return pd.Series(index.dayofyear.map(by_doy).to_numpy(), index=index)


MODELS = {
    "Media": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=seed),
    "LASSO": Lasso(alpha=0.001, random_state=seed, max_iter=5000),
    "Elastic Net": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=seed, max_iter=5000),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=seed, n_jobs=-1
    ),
    "HistGB": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=seed),
    "XGBoost": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
    ),
}

## 3. Validación cruzada temporal

`TimeSeriesSplit` con `gap`: cada partición ajusta con un prefijo de la serie y
evalúa con el tramo siguiente, dejando un hueco en medio. Es la única variante
válida aquí, y merece decirse por qué las alternativas no lo son.

Un `KFold` aleatorio pone instantes de las 14:00 en test y los de las 13:00 y las
15:00 en train. Con la autocorrelación medida en `01`, eso equivale a dar al
modelo la respuesta interpolada, y el resultado es un R² excelente que no
sobrevive a ningún dato nuevo. El `gap` extiende la misma precaución dentro de
cada partición.

Que las particiones crezcan tiene un efecto secundario que hay que tener presente
al leer la desviación típica de la tabla: la primera se ajusta con muchos menos
datos que la última, así que parte de la varianza entre particiones es tamaño de
muestra y no inestabilidad del modelo.

In [ ]:
cv = TimeSeriesSplit(n_splits=CV_SPLITS, gap=CV_GAP)

fig, ax = plt.subplots(figsize=(11, 2.6))
for fold, (fit_rows, score_rows) in enumerate(cv.split(train)):
    ax.plot(
        train.index[fit_rows],
        np.full(len(fit_rows), fold),
        lw=6,
        color=COLOR_NAMES["cornflower blue"],
        solid_capstyle="butt",
    )
    ax.plot(
        train.index[score_rows],
        np.full(len(score_rows), fold),
        lw=6,
        color=COLOR_NAMES["crimson"],
        solid_capstyle="butt",
    )
ax.set_yticks(range(CV_SPLITS), [f"fold {i}" for i in range(CV_SPLITS)])
ax.invert_yaxis()
ax.set_title(f"Validación cruzada temporal: azul ajusta, rojo evalúa, hueco de {CV_GAP} pasos")
savefig(fig, f"{FIG}/cv_scheme.png")

In [ ]:
def cross_validate(models: dict, block: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    features = block[columns]
    target = block[TARGET]
    records = []

    for fold, (fit_rows, score_rows) in enumerate(cv.split(block)):
        fit_features, fit_target = features.iloc[fit_rows], target.iloc[fit_rows]
        score_target = target.iloc[score_rows]

        # Baselines temporales: sin ajuste, pero con las mismas particiones y las
        # mismas métricas, que es lo que los hace comparables.
        shifted = persistence(target).iloc[score_rows]
        observed = shifted.notna()
        records.append(
            {
                "modelo": "Persistencia",
                "fold": fold,
                "segundos": 0.0,
                **wave_metrics(score_target[observed], shifted[observed]),
            }
        )
        records.append(
            {
                "modelo": "Climatología",
                "fold": fold,
                "segundos": 0.0,
                **wave_metrics(score_target, climatology(fit_target, score_target.index)),
            }
        )

        for name, model in models.items():
            started = time.perf_counter()
            fitted = model.fit(fit_features, fit_target)
            elapsed = time.perf_counter() - started
            predicted = fitted.predict(features.iloc[score_rows])
            records.append(
                {
                    "modelo": name,
                    "fold": fold,
                    "segundos": elapsed,
                    **wave_metrics(score_target, predicted),
                }
            )

    return pd.DataFrame(records)


folds = cross_validate(MODELS, train, predictors)
folds.head()

In [ ]:
metric_names = ["RMSE", "MAE", "R2", "Bias", "SI", "HH", "Pearson"]
summary = folds.groupby("modelo")[[*metric_names, "segundos"]].agg(["mean", "std"])
summary.columns = [f"{metric}_{statistic}" for metric, statistic in summary.columns]
summary = summary.sort_values(f"{DECIDE_BY}_mean")

# La desviación típica entre particiones importa tanto como la media: un modelo
# algo peor y estable suele ser preferible a uno mejor de media que se descuelga
# en una partición, porque esa partición es un año concreto del clima real.
summary[[f"{name}_mean" for name in metric_names] + ["RMSE_std", "segundos_mean"]].round(4)

In [ ]:
order = summary.index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, metric in zip(axes, ["RMSE", "SI", "Pearson"], strict=True):
    ax.boxplot(
        [folds.loc[folds["modelo"] == name, metric] for name in order],
        tick_labels=order,
        vert=False,
    )
    ax.set_xlabel(metric)
    ax.invert_yaxis()

baseline = summary.loc["Persistencia", "RMSE_mean"]
axes[0].axvline(baseline, color=COLOR_NAMES["crimson"], ls="--", label="persistencia")
axes[0].legend()
fig.suptitle("Dispersión entre particiones (mejor arriba)")
savefig(fig, f"{FIG}/cv_comparison.png")

beaten = summary.index[summary["RMSE_mean"] < baseline].tolist()
print(f"Modelos que baten a la persistencia: {beaten or 'ninguno'}")

## 4. Elección

El criterio es el RMSE medio en validación cruzada, con una condición previa: si
un modelo no bate a la persistencia, no se elige aunque sea el mejor de la tabla.
En ese caso lo que hay que revisar es el diseño de variables de `02`, no la lista
de modelos.

In [ ]:
trainable = [name for name in summary.index if name in MODELS]
best_name = summary.loc[trainable, f"{DECIDE_BY}_mean"].idxmin()
best_model = MODELS[best_name]

improvement = 100 * (baseline - summary.loc[best_name, "RMSE_mean"]) / baseline
print(f"Elegido: {best_name}")
print(f"RMSE en CV: {summary.loc[best_name, 'RMSE_mean']:.4f}")
print(f"Mejora sobre la persistencia: {improvement:+.1f}%")
if improvement <= 0:
    print("\nADVERTENCIA: no se bate a la persistencia. Revisar 02 antes de seguir.")

## 5. Test, una sola vez

El modelo elegido se reajusta con `train` + `valid` -- ya no hace falta reservar
validación, porque la decisión está tomada -- y se evalúa en `test`. Este número
es el que se publica.

A partir de aquí, cualquier cambio en el modelo invalida esta medida. Si después
de ver el resultado apeteciera probar otra cosa, hay que hacerlo con validación
cruzada y volver a esta sección al final: mirar test, ajustar y volver a mirar es
ajustar sobre test lentamente.

In [ ]:
development = pd.concat([train, valid]).sort_index()
final_model = best_model.fit(development[predictors], development[TARGET])

predicted = pd.Series(final_model.predict(test[predictors]), index=test.index)
observed = test[TARGET]

reference = {
    "Persistencia": persistence(matrix[TARGET]).reindex(test.index),
    "Climatología": climatology(development[TARGET], test.index),
}

rows = [{"modelo": best_name, **wave_metrics(observed, predicted)}]
for name, series in reference.items():
    usable = series.notna()
    rows.append({"modelo": name, **wave_metrics(observed[usable], series[usable])})

test_scores = pd.DataFrame(rows).set_index("modelo")
write_table(test_scores.reset_index(), TABLES / "test_scores.csv")
write_table(summary.reset_index(), TABLES / "cv_scores.csv")
test_scores.round(4)

## 6. Diagnóstico de residuos

Una tabla de métricas dice cuánto se falla; estas figuras dicen **dónde**, y esa
es la parte que se puede arreglar. Cuatro paneles:

1. **Observado frente a predicho.** Un modelo bien calibrado se pega a la
   diagonal. Lo habitual en oleaje es ver la nube doblarse por debajo en los
   valores altos: el modelo subestima los temporales.
2. **Residuo frente a predicción.** Un abanico que se abre es heterocedasticidad:
   el error crece con la magnitud, y las métricas globales están dominadas por los
   valores grandes.
3. **QQ del residuo.** No porque haga falta normalidad -- ningún modelo de aquí la
   supone -- sino porque las colas dicen si los errores grandes son más frecuentes
   de lo que una tabla de RMSE deja intuir.
4. **Error por estación.** Si el fallo se concentra en invierno, el problema son
   los temporales y no el modelo en general.

In [ ]:
from scipy import stats

residual = predicted - observed
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

limits = [min(observed.min(), predicted.min()), max(observed.max(), predicted.max())]
axes[0, 0].scatter(observed, predicted, s=3, alpha=0.15, color=COLOR_NAMES["cornflower blue"])
axes[0, 0].plot(limits, limits, color=COLOR_NAMES["crimson"], lw=1)
axes[0, 0].set_xlabel(f"{TARGET} observado")
axes[0, 0].set_ylabel(f"{TARGET} predicho")
axes[0, 0].set_title("Observado frente a predicho")

axes[0, 1].scatter(predicted, residual, s=3, alpha=0.15, color=COLOR_NAMES["cornflower blue"])
axes[0, 1].axhline(0, color=COLOR_NAMES["crimson"], lw=1)
axes[0, 1].set_xlabel("predicho")
axes[0, 1].set_ylabel("residuo")
axes[0, 1].set_title("Residuo frente a predicción")

stats.probplot(residual, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("QQ del residuo")

SEASONS = {
    12: "DJF",
    1: "DJF",
    2: "DJF",
    3: "MAM",
    4: "MAM",
    5: "MAM",
    6: "JJA",
    7: "JJA",
    8: "JJA",
    9: "SON",
    10: "SON",
    11: "SON",
}
ORDER = ["DJF", "MAM", "JJA", "SON"]
season = pd.Series(test.index.month.map(SEASONS), index=test.index)
axes[1, 1].boxplot([residual[season == name] for name in ORDER], tick_labels=ORDER)
axes[1, 1].axhline(0, color=COLOR_NAMES["crimson"], lw=1)
axes[1, 1].set_title("Residuo por estación")

fig.suptitle(f"Diagnóstico en test: {best_name}")
savefig(fig, f"{FIG}/residual_diagnostics.png")

### 6.1 Error por régimen

La figura que decide si el modelo sirve. Las métricas globales están dominadas por
el estado del mar más frecuente, que es el suave; el interés operativo está en el
percentil alto. Un modelo con un SI excelente en promedio y un sesgo de -30% por
encima del percentil 95 no es un buen modelo de oleaje: es un buen modelo de
calma.

In [ ]:
bins = [0, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
edges = observed.quantile(bins).to_numpy()
labels = [f"P{100 * bins[i]:g}-P{100 * bins[i + 1]:g}" for i in range(len(bins) - 1)]
regime = pd.cut(
    observed, np.unique(edges), labels=labels[: len(np.unique(edges)) - 1], include_lowest=True
)

by_regime = pd.DataFrame(
    [
        {
            "regimen": str(name),
            "n": len(block),
            **wave_metrics(observed[block.index], predicted[block.index]),
        }
        for name, block in observed.groupby(regime, observed=True)
    ]
).set_index("regimen")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, metric in zip(axes, ["RMSE", "Bias", "SI"], strict=True):
    colours = [
        COLOR_NAMES["crimson"] if value < 0 else COLOR_NAMES["cornflower blue"]
        for value in by_regime[metric]
    ]
    ax.bar(by_regime.index, by_regime[metric], color=colours)
    ax.axhline(0, color=NEUTRALS["dark_slate"], lw=0.8)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=45)
fig.suptitle(f"Error por régimen de {TARGET} (percentiles del observado)")
savefig(fig, f"{FIG}/error_by_regime.png")

by_regime.round(4)

In [ ]:
destination = settings.paths.models / f"{best_name.lower().replace(' ', '_')}.joblib"
joblib.dump(
    {
        "model": final_model,
        "name": best_name,
        "predictors": predictors,
        "target": TARGET,
        "test_scores": test_scores.loc[best_name].to_dict(),
        "params": {
            "USE_SELECTED": USE_SELECTED,
            "CV_SPLITS": CV_SPLITS,
            "CV_GAP": CV_GAP,
            "DECIDE_BY": DECIDE_BY,
        },
    },
    destination,
)
# 05 abre este fichero por su nombre, así que se deja anotado en un sitio fijo en
# lugar de dejar que el siguiente notebook adivine cuál es el modelo elegido.
(settings.paths.models / "best_model.txt").write_text(destination.name + "\n")
print(destination)

## Conclusiones

A rellenar antes de pasar a `05`:

1. **Qué modelo gana y por cuánto**, sobre la persistencia y no sobre la media.
2. **Si la mejora justifica la complejidad.** Un XGBoost que gana un 2% a una
   Ridge suele no compensar: la Ridge se explica en una frase, se reentrena en
   segundos y no tiene hiperparámetros que revisar cada año.
3. **Dónde falla** (§6 y §6.1), y muy en concreto qué sesgo tiene por encima del
   percentil 95.
4. **Si el error se concentra en alguna estación**, porque eso apunta a variables
   que faltan y no a un modelo mal elegido.

Lo que queda pendiente, y en qué orden:

- **Ajuste de hiperparámetros.** Los de arriba son los valores por defecto
  razonables, deliberadamente: sirven para ordenar familias de modelos, no para
  dar el mejor número posible. El ajuste va después de esta comparación y con la
  misma validación cruzada temporal -- nunca con test.
- **`05_interpretability.ipynb`**, sobre el modelo que se acaba de guardar.
- Cuando el modelo elegido deje de cambiar, `wave_metrics` pasa a
  `src/packagename/models/metrics.py` y el entrenamiento a
  `src/packagename/cli/train.py`, que ya existe y ya registra la configuración
  resuelta en W&B.